# Compose sport datasets

Run `nb01.07.01` to import sources and `nb01.07.02` to review duplicates first.
This notebook reads `sports_objects_master` and exports four versioned COCO
datasets. Samples tagged `review:*duplicate` are excluded; source annotations,
images, and review decisions remain unchanged.

Each export lives at `DATA_ROOT/composed/<target-name>/coco/`, with
`annotations/instances_{train,val,test}.json` and `images/{train,val,test}/`.
Images use their cached `pixel_digest` plus the original extension. Media is
hardlinked when possible, with a copy fallback. Existing targets stop the build.

`Dataset(root).coco` returns the COCO directory. `Dataset(root).yolo` creates
a matching YOLO export on first access, then reuses it. Keep each version's
COCO content fixed so both formats stay consistent.

In [1]:
from fnmatch import fnmatchcase
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import cast

import datumaro as dm
import fiftyone as fo
import pandas as pd
from detection_common import configure_stdio_relative_path
from IPython.display import display

from dataset_builder import Dataset, prefer_hardlinked_datumaro_media
from dataset_builder.config import DATA_ROOT, WORKSPACE_ROOT
from dataset_builder.datumaro import ExportFormat

configure_stdio_relative_path(WORKSPACE_ROOT)
COMPOSED_ROOT = DATA_ROOT / "composed"
SPLITS = ("train", "val", "test")

## 1. Select retained images

Filtering uses existing review tags only. Basketball, Gaelic football, and
tennis keep their original splits (`valid` becomes `val`). Football uses the
fixed source assignments shown below.

In [2]:
master = fo.load_dataset("sports_objects_master")
duplicate_tags = [
    tag for tag in cast(list[str], master.distinct("tags"))
    if fnmatchcase(tag, "review:*duplicate")
]
retained_view = (
    master.match_tags(duplicate_tags, bool=False) if duplicate_tags else master.view()
)
print(f"Retained images: {retained_view.count():,} / {master.count():,}")

Retained images: 28,176 / 34,003


### Shared export helper

Check source labels before exporting, then build independent Datumaro items.
A mapping value of `None` removes an annotation; an image with no remaining
annotations is kept as background. Whole-image exclusions run first.

In [3]:
def export_dataset(
    collection, name, label_maps, *, classes=None, split_by_source=None,
    exclude_images=None,
):
    """
    Export mapped detections as canonical COCO without modifying the collection.
    """
    root = COMPOSED_ROOT / name
    if root.exists():
        raise FileExistsError(f"Target already exists: {root}")
    view = collection.match(fo.ViewField("source_dataset").is_in(list(label_maps)))
    exclude_images = exclude_images or {}
    inventory = []
    for source, mapping in label_maps.items():
        source_view = view.match(fo.ViewField("source_dataset") == source)
        count = source_view.count()
        if not count:
            continue
        labels = sorted(source_view.distinct("original_labels.detections.label"))
        inventory.append({"source": source, "images": count, "labels": labels})
        if mapping is not None:
            unknown = set(labels) - set(mapping) - set(exclude_images.get(source, ()))
            if unknown:
                raise ValueError(f"Unexpected labels in {source}: {sorted(unknown)}")
    display(pd.DataFrame(inventory))
    classes = classes or sorted(view.distinct("original_labels.detections.label"))
    if not classes:
        raise ValueError(f"No labels available for {name}")
    label_ids = {label: index for index, label in enumerate(classes)}
    items, records, seen_digests = [], [], set()
    fields = ["source_dataset", "original_split", "pixel_digest", "metadata",
              "original_labels"]
    for sample in view.select_fields(fields).iter_samples():
        source = sample.source_dataset
        split = (split_by_source or {}).get(source, sample.original_split)
        split = "val" if split == "valid" else split
        if split not in SPLITS:
            raise ValueError(f"Unknown split for {source}: {split!r}")
        detections = sample.original_labels.detections if sample.original_labels else []
        excluded = any(d.label in exclude_images.get(source, ()) for d in detections)
        record = {"source": source, "split": split, "images": 0,
                  "annotations": 0, "backgrounds": 0, "excluded_images": int(excluded)}
        records.append(record)
        if excluded:
            continue
        path = Path(sample.filepath)
        digest = sample.pixel_digest
        if (
            not digest or len(digest) != 64
            or any(c not in "0123456789abcdef" for c in digest)
        ):
            raise ValueError(f"Missing or invalid pixel_digest: {path}")
        filename = digest + path.suffix
        if digest in seen_digests:
            raise ValueError(f"Duplicate pixel_digest: {digest}")
        seen_digests.add(digest)
        width, height = sample.metadata.width, sample.metadata.height
        mapping, annotations = label_maps[source], []
        for detection in detections:
            label = detection.label if mapping is None else mapping[detection.label]
            if label is None:
                continue
            x, y, w, h = detection.bounding_box
            annotations.append(dm.Bbox(
                x * width, y * height, w * width, h * height,
                label=label_ids[label],
                attributes={
                    "is_crowd": bool(detection.get_attribute_value("iscrowd", 0))
                },
            ))
        items.append(dm.DatasetItem(
            id=digest, subset=split,
            media=dm.Image.from_file(str(path), size=(height, width)),
            annotations=annotations,
        ))
        record.update(
            images=1, annotations=len(annotations), backgrounds=int(not annotations)
        )
    dataset = dm.Dataset.from_iterable(items, categories=classes)
    COMPOSED_ROOT.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(prefix=f".{name}-", dir=COMPOSED_ROOT) as temporary:
        staging = Path(temporary) / name
        with prefer_hardlinked_datumaro_media():
            dataset.export(str(staging / "coco"), format=ExportFormat.COCO_INSTANCES,
                           save_media=True, reindex=True)
        staging.rename(root)
    summary = pd.DataFrame(records).groupby(["source", "split"]).sum()
    display(summary)
    totals = summary.groupby("split").sum().reindex(SPLITS, fill_value=0)
    totals["image_percent"] = (100 * totals["images"] / totals["images"].sum()).round(2)
    display(totals)
    result = Dataset(root)
    print(f"COCO export: {result.coco}")
    return result

## 2. Basketball — `basketball-v0.1`

Combine `basketball-lokg2` and `basketball-xil7x`, preserving original splits.
Source `ball-tracker-jpjs8` is excluded.

| Original labels | Target label |
| --- | --- |
| `ball`, `basketball` | `basketball` |
| `people`, `human` | `person` |
| `rim` | `rim` |

In [4]:
basketball_labels = {
    "ball": "basketball", "basketball": "basketball",
    "people": "person", "human": "person", "rim": "rim",
}
basketball = export_dataset(
    retained_view, "basketball-v0.1",
    {source: basketball_labels for source in ("basketball-lokg2", "basketball-xil7x")},
    classes=["basketball", "person", "rim"],
)

,source,images,labels
0,basketball-lokg2,771,"[basketball, people, rim]"
1,basketball-xil7x,7083,"[ball, human, rim]"


images  annotations  backgrounds  excluded_images
source           split                                                   
basketball-lokg2 test       37           90            0                0
                 train     658         1289            0                0
                 val        76          167            0                0
basketball-xil7x test      830         3216            0                0
                 train    5072        20226            0                0
                 val      1181         4773            0                0

,images,annotations,backgrounds,excluded_images,image_percent
split,,,,,
train,5730,21515,0,0,72.96
val,1257,4940,0,0,16.00
test,867,3306,0,0,11.04


COCO export: data/composed/basketball-v0.1/coco


## 3. Football — `football-v0.1`

Label matching is case-sensitive. Remove an entire image when `football-csdy0`
contains `otherball` or `football-tnb17` contains `other`. Other removals below
affect annotations only; keep the image even when it becomes background.

| Source | Labels mapped to `football` | Other rules |
| --- | --- | --- |
| `football-2k4ty` | `football` | |
| `football-5beob` | `ball` | Keep `person` |
| `football-ablqb` | `football` | |
| `football-csdy0` | `football` | Exclude images with `otherball` |
| `football-detection-ftt4q` | `Football` | |
| `football-detection-gjrtq` | `Football`, `ball` | `Player`, `referee` → `person`; remove `big` |
| `football-gsgma` | `football`, `soccer`, `soccer ball`, `soccer_ball` | Remove `football2` |
| `football-rmhvy` | `Football`, `football_1` | Remove `Not football`, `hball` |
| `football-tnb17` | `football` | Exclude images with `other` |
| `football-wsef2` | `football` | |
| `football-ywyvk` | `Football` | |
| `tracer-football` | `Soccer ball`, `football`, `soccer` | Remove `ball`, `other`, `others` |

In [5]:
football_labels = {
    "football-2k4ty": {"football": "football"},
    "football-5beob": {"ball": "football", "person": "person"},
    "football-ablqb": {"football": "football"},
    "football-csdy0": {"football": "football"},
    "football-detection-ftt4q": {"Football": "football"},
    "football-detection-gjrtq": {
        "Football": "football", "ball": "football",
        "Player": "person", "referee": "person", "big": None,
    },
    "football-gsgma": {
        "football": "football", "soccer": "football", "soccer ball": "football",
        "soccer_ball": "football", "football2": None,
    },
    "football-rmhvy": {
        "Football": "football", "football_1": "football",
        "Not football": None, "hball": None,
    },
    "football-tnb17": {"football": "football"},
    "football-wsef2": {"football": "football"},
    "football-ywyvk": {"Football": "football"},
    "tracer-football": {
        "Soccer ball": "football", "football": "football", "soccer": "football",
        "ball": None, "other": None, "others": None,
    },
}

### Fixed source assignments

As in `nb01.03`, each source belongs wholly to one target split. Combine its
original partitions using these fixed choices. After filtering, the planned
totals are 3,288 / 697 / 685 images, approximately 70.41:14.93:14.67.

| Source | Retained images | Target split |
| --- | ---: | --- |
| `football-detection-ftt4q` | 799 | train |
| `football-detection-gjrtq` | 592 | train |
| `football-gsgma` | 663 | train |
| `football-rmhvy` | 1,234 | train |
| `football-2k4ty` | 191 | val |
| `football-ablqb` | 100 | val |
| `football-csdy0` | 81 | val |
| `football-ywyvk` | 325 | val |
| `football-5beob` | 28 | test |
| `football-tnb17` | 43 | test |
| `football-wsef2` | 133 | test |
| `tracer-football` | 481 | test |

In [6]:
football_splits = {
    "football-detection-ftt4q": "train", "football-detection-gjrtq": "train",
    "football-gsgma": "train", "football-rmhvy": "train",
    "football-2k4ty": "val", "football-ablqb": "val",
    "football-csdy0": "val", "football-ywyvk": "val",
    "football-5beob": "test", "football-tnb17": "test",
    "football-wsef2": "test", "tracer-football": "test",
}
football = export_dataset(
    retained_view, "football-v0.1", football_labels,
    classes=["football", "person"], split_by_source=football_splits,
    exclude_images={"football-csdy0": {"otherball"}, "football-tnb17": {"other"}},
)

,source,images,labels
0,football-2k4ty,191,[football]
1,football-5beob,28,"[ball, person]"
2,football-ablqb,100,[football]
3,football-csdy0,147,"[football, otherball]"
4,football-detection-ftt4q,799,[Football]
5,football-detection-gjrtq,592,"[Football, Player, ball, big, referee]"
6,football-gsgma,663,"[football, football2, soccer, soccer ball, soc..."
7,football-rmhvy,1234,"[Football, Not football, football_1, hball]"
8,football-tnb17,197,"[football, other]"
9,football-wsef2,133,[football]


,,images,annotations,backgrounds,excluded_images
source,split,,,,
football-2k4ty,val,191,204,0,0
football-5beob,test,28,108,0,0
football-ablqb,val,100,100,0,0
football-csdy0,val,81,149,0,66
football-detection-ftt4q,train,799,838,4,0
football-detection-gjrtq,train,592,578,215,0
football-gsgma,train,663,721,4,0
football-rmhvy,train,1234,2026,3,0
football-tnb17,test,43,59,0,154


,images,annotations,backgrounds,excluded_images,image_percent
split,,,,,
train,3288,4163,226,0,70.41
val,697,838,0,66,14.93
test,685,936,0,154,14.67


COCO export: data/composed/football-v0.1/coco


## 4. Gaelic football — `gaelic_football-v0.1`

Export retained samples from `coco_gaelic_football`, preserving their current
label names, boxes, and original splits. No label remapping is applied.

In [7]:
gaelic_football = export_dataset(
    retained_view, "gaelic_football-v0.1", {"coco_gaelic_football": None},
)

,source,images,labels
0,coco_gaelic_football,117,[gaelic_football]


images  annotations  backgrounds  excluded_images
source               split                                                   
coco_gaelic_football test       18           16            4                0
                     train      82           76           20                0
                     val        17           15            5                0

,images,annotations,backgrounds,excluded_images,image_percent
split,,,,,
train,82,76,20,0,70.09
val,17,15,5,0,14.53
test,18,16,4,0,15.38


COCO export: data/composed/gaelic_football-v0.1/coco


## 5. Tennis — `tennis-v0.1`

Export retained samples from `tennis-racket-r6mgq`, preserving their current
label names, boxes, and original splits. No label remapping is applied.

In [8]:
tennis = export_dataset(
    retained_view, "tennis-v0.1", {"tennis-racket-r6mgq": None},
)

,source,images,labels
0,tennis-racket-r6mgq,15311,"[Racket, ball, object, overlap]"


images  annotations  backgrounds  excluded_images
source              split                                                   
tennis-racket-r6mgq test      437          548          107                0
                    train   14259        19200         1884                0
                    val       615          924          103                0

,images,annotations,backgrounds,excluded_images,image_percent
split,,,,,
train,14259,19200,1884,0,93.13
val,615,924,103,0,4.02
test,437,548,107,0,2.85


COCO export: data/composed/tennis-v0.1/coco


## Use an export

Each returned object is a `dataset_builder.Dataset`. Use `football.coco` for
the canonical COCO path. Request `football.yolo` when YOLO is needed; the first
access converts COCO and subsequent accesses reuse the existing directory.
A YOLO training configuration is then at `football.yolo / "data.yaml"`.